In [1]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory:", os.getcwd())

Working directory: /groups/nils/members/andras/scrna_project


In [2]:
import sqlite3
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

DB_PATH = "results/variants.db"

def query(sql, params=()):
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(sql, conn, params=params)

print("Connected to:", DB_PATH)
from IPython.display import display, clear_output


Connected to: results/variants.db


## Cross-sample QC overview
Mapping rate, variant counts, and mean depth for every run in the database.

In [3]:
runs = query("""
    SELECT r.run_id, s.name AS sample, s.cell_line,
           r.mapping_rate, r.total_raw, r.total_filt, r.run_date
    FROM runs r
    JOIN samples s ON r.sample_id = s.sample_id
    ORDER BY s.cell_line, r.run_id
""")

if runs.empty:
    print("No runs found in database.")
else:
    runs["label"] = runs["cell_line"].fillna(runs["sample"])
    runs_sorted = runs.sort_values("mapping_rate", ascending=True)

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Mapping rate (%) — all runs", "Filtered variants per run"),
        horizontal_spacing=0.12
    )

    # Horizontal bar — mapping rate, coloured by pass/fail
    colors = ["#e63946" if v < 85 else "#2d6a4f" for v in runs_sorted["mapping_rate"]]
    fig.add_trace(go.Bar(
        y=runs_sorted["label"],
        x=runs_sorted["mapping_rate"],
        orientation="h",
        marker_color=colors,
        hovertemplate="%{y}<br>Mapping rate: %{x:.1f}%<extra></extra>"
    ), row=1, col=1)
    fig.add_vline(x=85, line_dash="dash", line_color="#e63946",
                  annotation_text="85%", row=1, col=1)

    # Horizontal bar — filtered variants
    runs_sorted2 = runs.sort_values("total_filt", ascending=True)
    fig.add_trace(go.Bar(
        y=runs_sorted2["label"],
        x=runs_sorted2["total_filt"],
        orientation="h",
        marker_color="#457b9d",
        hovertemplate="%{y}<br>Filtered variants: %{x:,}<extra></extra>"
    ), row=1, col=2)

    n = len(runs)
    fig.update_layout(
        height=max(400, 22 * n),
        title=f"Run QC overview — {n} runs",
        showlegend=False
    )
    fig.update_yaxes(tickmode='linear', dtick=1)
    fig.update_xaxes(range=[0, 105], row=1, col=1)
    fig.show()

    # Summary table grouped by cell line
    summary = (
        runs.groupby("label").agg(
            n_runs=("run_id", "count"),
            mean_mapping=("mapping_rate", "mean"),
            mean_filt_vars=("total_filt", "mean"),
        ).reset_index().rename(columns={"label": "cell_line"})
        .sort_values("mean_filt_vars", ascending=False)
    )
    display(summary.style.format({
        "mean_mapping": "{:.1f}",
        "mean_filt_vars": "{:,.0f}"
    }))


,cell_line,n_runs,mean_mapping,mean_filt_vars
8,DiFi,1,100.0,"66,114"
19,HT55,1,100.0,"60,043"
20,IS1,1,100.0,"58,557"
41,V9P,1,100.0,"56,975"
27,LIM2405,1,100.0,"56,924"
2,C70,1,100.0,"52,622"
31,NCIH747,1,100.0,"51,420"
30,LS513,1,100.0,"51,376"
16,HRA19,1,100.0,"50,874"
15,HCT8,1,100.0,"50,428"


## Per-run deep-dive
Select a run to explore depth distribution, allele frequencies, QUAL scores, and chromosomal variant distribution.

In [4]:
run_ids = runs['run_id'].tolist() if not runs.empty else []

run_selector = widgets.Dropdown(
    options=run_ids,
    description='Run:',
    layout=widgets.Layout(width='350px')
)

_perrun_busy = [False]

def render_perrun(run_id):
    if _perrun_busy[0]:
        return
    _perrun_busy[0] = True
    try:
        clear_output(wait=False)
        display(run_selector)

        calls = query("""
            SELECT gc.depth, gc.quality, gc.allele_freq,
                   v.variant_type, v.chromosome
            FROM genotype_calls gc
            JOIN variants v ON gc.variant_id = v.variant_id
            WHERE gc.run_id = ?
        """, (run_id,))

        if calls.empty:
            print(f'No variant calls found for {run_id}.')
            return

        main_chroms = [f'chr{i}' for i in list(range(1, 23)) + ['X', 'Y', 'M']]
        calls_main = calls[calls['chromosome'].isin(main_chroms)]
        type_counts = calls['variant_type'].value_counts().reset_index()
        type_counts.columns = ['type', 'count']

        fig = make_subplots(
            rows=2, cols=3,
            subplot_titles=(
                'Read depth distribution', 'QUAL score distribution', 'Variant types',
                'Allele frequency spectrum', 'Variants per chromosome', 'Depth vs QUAL'
            ),
            specs=[
                [{'type': 'xy'}, {'type': 'xy'}, {'type': 'domain'}],
                [{'type': 'xy'}, {'type': 'xy'}, {'type': 'xy'}]
            ],
            horizontal_spacing=0.10, vertical_spacing=0.15
        )

        depth_cap = calls['depth'].quantile(0.99)
        fig.add_trace(go.Histogram(
            x=calls[calls['depth'] <= depth_cap]['depth'],
            nbinsx=50, marker_color='#457b9d', name='Depth'
        ), row=1, col=1)
        fig.add_trace(go.Histogram(
            x=calls['quality'], nbinsx=50,
            marker_color='#2d6a4f', name='QUAL'
        ), row=1, col=2)
        fig.add_trace(go.Pie(
            labels=type_counts['type'],
            values=type_counts['count'],
            hole=0.4,
            marker_colors=['#2d6a4f', '#e63946', '#457b9d', '#f4a261']
        ), row=1, col=3)
        fig.add_trace(go.Histogram(
            x=calls['allele_freq'], nbinsx=40,
            marker_color='#f4a261', name='AF'
        ), row=2, col=1)

        chrom_counts = (
            calls_main.groupby('chromosome').size()
            .reindex(main_chroms).fillna(0).reset_index()
        )
        chrom_counts.columns = ['chromosome', 'count']
        fig.add_trace(go.Bar(
            x=chrom_counts['chromosome'], y=chrom_counts['count'],
            marker_color='#6d6875', name='Variants'
        ), row=2, col=2)

        sample_n = min(5000, len(calls))
        scatter_df = calls.sample(sample_n, random_state=42)
        fig.add_trace(go.Scatter(
            x=scatter_df['depth'], y=scatter_df['quality'],
            mode='markers',
            marker=dict(size=3, opacity=0.4, color='#1d3557'),
            name='calls'
        ), row=2, col=3)

        fig.update_layout(
            height=700,
            title=f'Per-run deep-dive: {run_id}  ({len(calls):,} variant calls)',
            showlegend=False
        )
        fig.update_xaxes(title_text='Depth', row=1, col=1)
        fig.update_xaxes(title_text='QUAL', row=1, col=2)
        fig.update_xaxes(title_text='Allele frequency', row=2, col=1)
        fig.update_xaxes(tickangle=45, row=2, col=2)
        fig.update_xaxes(title_text='Depth', row=2, col=3)
        fig.update_yaxes(title_text='QUAL', row=2, col=3)
        display(fig)

        stats = pd.DataFrame([{
            'total_calls': len(calls),
            'median_depth': round(calls['depth'].median(), 1),
            'mean_depth': round(calls['depth'].mean(), 1),
            'median_qual': round(calls['quality'].median(), 1),
            'pct_af_gt50': round((calls['allele_freq'] > 0.5).mean() * 100, 1)
        }])
        display(stats)
    finally:
        _perrun_busy[0] = False

run_selector.observe(lambda ch: render_perrun(ch['new']), names='value')
render_perrun(run_ids[0])


Dropdown(description='Run:', layout=Layout(width='350px'), options=('SRR5071654_hg38', 'SRR5071655_hg38', 'SRR…

,total_calls,median_depth,mean_depth,median_qual,pct_af_gt50
0,35361,26.0,52.5,148.4,28.3


## Cross-sample comparison
Select two runs to compare their QC metrics side-by-side and see how many variants they share.

In [5]:
sel_a = widgets.Dropdown(
    options=run_ids, description='Run A:',
    layout=widgets.Layout(width='350px')
)
sel_b = widgets.Dropdown(
    options=run_ids,
    value=run_ids[1] if len(run_ids) > 1 else run_ids[0],
    description='Run B:',
    layout=widgets.Layout(width='350px')
)

_compare_busy = [False]

def render_compare(run_a, run_b):
    if _compare_busy[0]:
        return
    _compare_busy[0] = True
    try:
        clear_output(wait=False)
        display(widgets.HBox([sel_a, sel_b]))

        def get_variants(rid):
            return query("""
                SELECT v.chromosome, v.position, v.ref_allele, v.alt_allele,
                       gc.depth, gc.allele_freq, gc.quality
                FROM genotype_calls gc
                JOIN variants v ON gc.variant_id = v.variant_id
                WHERE gc.run_id = ?
            """, (rid,))

        df_a = get_variants(run_a)
        df_b = get_variants(run_b)

        if df_a.empty or df_b.empty:
            print('One or both runs have no variant data.')
            return

        key_cols = ['chromosome', 'position', 'ref_allele', 'alt_allele']
        set_a = set(df_a[key_cols].itertuples(index=False, name=None))
        set_b = set(df_b[key_cols].itertuples(index=False, name=None))
        shared = len(set_a & set_b)
        only_a = len(set_a - set_b)
        only_b = len(set_b - set_a)

        fig = make_subplots(
            rows=1, cols=3,
            subplot_titles=('Variant overlap', 'Depth comparison', 'AF comparison'),
            horizontal_spacing=0.12
        )

        labels = [f'{run_a} only', 'Shared', f'{run_b} only']
        values = [only_a, shared, only_b]
        fig.add_trace(go.Bar(
            x=labels, y=values,
            marker_color=['#457b9d', '#2d6a4f', '#e63946'],
            text=[f'{v:,}' for v in values], textposition='auto'
        ), row=1, col=1)

        depth_cap = max(df_a['depth'].quantile(0.99), df_b['depth'].quantile(0.99))
        fig.add_trace(go.Box(
            y=df_a[df_a['depth'] <= depth_cap]['depth'],
            name=run_a, marker_color='#457b9d', boxmean=True
        ), row=1, col=2)
        fig.add_trace(go.Box(
            y=df_b[df_b['depth'] <= depth_cap]['depth'],
            name=run_b, marker_color='#e63946', boxmean=True
        ), row=1, col=2)
        fig.add_trace(go.Histogram(
            x=df_a['allele_freq'], nbinsx=30,
            name=run_a, opacity=0.6, marker_color='#457b9d'
        ), row=1, col=3)
        fig.add_trace(go.Histogram(
            x=df_b['allele_freq'], nbinsx=30,
            name=run_b, opacity=0.6, marker_color='#e63946'
        ), row=1, col=3)

        overlap_pct = shared / len(set_a) * 100 if set_a else 0
        jaccard = shared / len(set_a | set_b) * 100 if (set_a | set_b) else 0
        fig.update_layout(
            height=420, barmode='overlay',
            title=f'{run_a}  vs  {run_b}  |  overlap {overlap_pct:.1f}%  |  Jaccard {jaccard:.1f}%'
        )
        display(fig)

        run_info = query("""
            SELECT r.run_id, s.name AS sample, r.mapping_rate, r.total_raw, r.total_filt
            FROM runs r JOIN samples s ON r.sample_id = s.sample_id
            WHERE r.run_id IN (?, ?)
        """, (run_a, run_b))
        display(run_info.style.format({'mapping_rate': '{:.1f}',
                                       'total_raw': '{:,}', 'total_filt': '{:,}'}))
    finally:
        _compare_busy[0] = False

def on_compare_change(change):
    render_compare(sel_a.value, sel_b.value)

sel_a.observe(on_compare_change, names='value')
sel_b.observe(on_compare_change, names='value')

if len(run_ids) >= 2:
    render_compare(run_ids[0], run_ids[1])


,run_id,sample,mapping_rate,total_raw,total_filt
0,SRR5071654_hg38,SRR5071654,100.0,"2,081,645","35,361"
1,SRR5071655_hg38,SRR5071655,100.0,"2,096,354","32,549"


## Variant overlap heatmap
Jaccard similarity between all cell lines — reveals which lines share variants and whether the DB coverage is well-distributed. Only ALT calls are compared.

In [6]:
import numpy as np
from scipy.stats import gaussian_kde as _gkde

alt_calls = query("""
    SELECT r.run_id, s.cell_line, v.chromosome, v.position, v.ref_allele, v.alt_allele
    FROM genotype_calls gc
    JOIN variants v ON gc.variant_id = v.variant_id
    JOIN runs r ON gc.run_id = r.run_id
    JOIN samples s ON r.sample_id = s.sample_id
    WHERE gc.genotype != '0/0'
""")

alt_calls["label"] = alt_calls["cell_line"].fillna(alt_calls["run_id"])
key_cols = ["chromosome", "position", "ref_allele", "alt_allele"]

grouped = {}
for label, grp in alt_calls.groupby("label"):
    grouped[label] = set(map(tuple, grp[key_cols].values))

labels = sorted(grouped.keys())
n = len(labels)
jaccard = np.zeros((n, n))
asymm   = np.zeros((n, n))

for i, a in enumerate(labels):
    for j, b in enumerate(labels):
        sa, sb = grouped[a], grouped[b]
        inter = len(sa & sb)
        union = len(sa | sb)
        jaccard[i, j] = inter / union if union else 0
        asymm[i, j]   = inter / len(sa) if sa else 0

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Jaccard similarity (symmetric)",
        "Containment — row % found in column (asymmetric)"
    ),
    horizontal_spacing=0.12
)

fig.add_trace(go.Heatmap(
    z=jaccard, x=labels, y=labels,
    colorscale="Blues", zmin=0, zmax=1,
    text=np.round(jaccard, 2), texttemplate="%{text}",
    hovertemplate="%{y} vs %{x}<br>Jaccard: %{z:.3f}<extra></extra>",
    showscale=True, colorbar=dict(x=0.44, len=0.9)
), row=1, col=1)

fig.add_trace(go.Heatmap(
    z=asymm, x=labels, y=labels,
    colorscale="Oranges", zmin=0, zmax=1,
    text=np.round(asymm, 2), texttemplate="%{text}",
    hovertemplate="%{y} → %{x}<br>%{z:.1%} of row's variants in column<extra></extra>",
    showscale=True, colorbar=dict(x=1.01, len=0.9)
), row=1, col=2)

fig.update_layout(
    title="Variant overlap — all ALT calls",
    height=max(500, 28 * n),
    width=max(900, 40 * n)
)
fig.update_xaxes(tickangle=45)
fig.show()

# ── Similarity distributions ──────────────────────────────────────────────
triu     = np.triu_indices(n, k=1)
jac_vals = jaccard[triu]

off_diag  = ~np.eye(n, dtype=bool)
asym_vals = asymm[off_diag]

x_grid = np.linspace(0, 1, 300)

fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"Jaccard similarity  ({len(jac_vals):,} pairs, upper triangle)",
        f"Containment  ({len(asym_vals):,} values, all off-diagonal)"
    ),
    horizontal_spacing=0.14
)

for col_idx, (vals, color) in enumerate(
    [(jac_vals, "#457b9d"), (asym_vals, "#2d6a4f")], start=1
):
    kde_y = _gkde(vals)(x_grid)
    med   = float(np.median(vals))

    fig2.add_trace(go.Histogram(
        x=vals, histnorm="probability density", nbinsx=40,
        marker_color=color, opacity=0.30, showlegend=False,
        hovertemplate="Value: %{x:.3f}<br>Density: %{y:.2f}<extra></extra>"
    ), row=1, col=col_idx)

    fig2.add_trace(go.Scatter(
        x=x_grid, y=kde_y, mode="lines",
        line=dict(color=color, width=2.5), showlegend=False,
        hovertemplate="Value: %{x:.3f}<br>Density: %{y:.2f}<extra></extra>"
    ), row=1, col=col_idx)

    fig2.add_trace(go.Scatter(
        x=[med, med], y=[0, kde_y.max() * 1.15],
        mode="lines", line=dict(color=color, width=1.5, dash="dot"),
        showlegend=False, hoverinfo="skip"
    ), row=1, col=col_idx)
    fig2.add_annotation(
        x=med, y=kde_y.max() * 1.18,
        text=f"median {med:.2f}", showarrow=False,
        font=dict(size=11, color=color),
        xref=f"x{'' if col_idx == 1 else col_idx}",
        yref=f"y{'' if col_idx == 1 else col_idx}"
    )

fig2.update_xaxes(range=[0, 1], title_text="Similarity")
fig2.update_yaxes(title_text="Density", row=1, col=1)
fig2.update_layout(height=380, title="Variant overlap — similarity distributions")
fig2.show()


---
## scRNA demultiplexing results
Visualisations for `final_assignments.tsv` produced by the scRNA demux pipeline (cellsnp-lite → Vireo → match_vireo → merge_demux).

In [7]:
import glob, os

demux_runs = sorted(
    os.path.basename(os.path.dirname(p))
    for p in glob.glob("results/demux/*/final_assignments.tsv")
)

demux_selector = widgets.Dropdown(
    options=demux_runs,
    description="Demux run:",
    layout=widgets.Layout(width="350px")
)
def load_demux(run_id):
    fa = pd.read_csv(f"results/demux/{run_id}/final_assignments.tsv", sep="\t")
    dm_path = f"results/demux/{run_id}/donor_matches.tsv"
    dm = pd.read_csv(dm_path, sep="\t") if os.path.exists(dm_path) else pd.DataFrame()
    return fa, dm

def render_scrna(run_id):
    fa, dm = load_demux(run_id)

    # ── 1. Cell composition bar ──────────────────────────────────────────
    singlets    = fa[fa["source"] == "db_match"].copy()
    vireo_only  = fa[fa["source"] == "vireo_only"].copy()

    # DB-matched lines sorted by count
    db_order = (
        singlets.groupby("cell_line").size()
        .sort_values(ascending=False).index.tolist()
    )
    # Unmatched vireo donors sorted by count
    vireo_order = (
        vireo_only.groupby("cell_line").size()
        .sort_values(ascending=False).index.tolist()
    )
    line_order = db_order + vireo_order  # used for doublet heatmap axes too

    fig1 = go.Figure()
    colors_conf = {"high": "#2d6a4f", "low": "#f4a261"}
    for conf in ["high", "low"]:
        sub = singlets[singlets["match_confidence"] == conf]
        counts = sub["cell_line"].value_counts().reindex(db_order, fill_value=0)
        fig1.add_trace(go.Bar(
            x=counts.index, y=counts.values,
            name=f"DB match ({conf})",
            marker_color=colors_conf[conf]
        ))

    # Unmatched vireo donors — show as a separate colour group
    if len(vireo_only):
        vo_counts = vireo_only["cell_line"].value_counts().reindex(vireo_order, fill_value=0)
        fig1.add_trace(go.Bar(
            x=vo_counts.index, y=vo_counts.values,
            name="vireo only (unmatched)",
            marker_color="#6d6875"
        ))

    for status, color in [("doublet", "#e63946"), ("unassigned", "#adb5bd")]:
        fig1.add_trace(go.Bar(
            x=[status], y=[(fa["source"] == status).sum()],
            name=status, marker_color=color
        ))

    fig1.update_layout(
        barmode="stack",
        title=f"Cell composition — {run_id}  ({len(fa):,} cells)",
        xaxis_title="Cell line / donor", yaxis_title="Cell count",
        height=420, legend_title="Assignment"
    )
    display(fig1)

    # ── 2. Doublet pair heatmap ───────────────────────────────────────────
    doublets = fa[fa["source"] == "doublet"].copy()
    dbl_mat = pd.DataFrame(0, index=line_order, columns=line_order)

    for pair_str in doublets["cell_line"].str.replace("^doublet:", "", regex=True):
        parts = [p.strip() for p in pair_str.split("+")]
        if len(parts) == 2:
            a, b = parts
            # Normalise: strip "vireo:" prefix for lookup but keep label
            if a in dbl_mat.index and b in dbl_mat.columns:
                dbl_mat.loc[a, b] += 1
                dbl_mat.loc[b, a] += 1

    dbl_vals = dbl_mat.values.astype(float)
    dbl_text = dbl_mat.values.astype(str)
    tril = np.tril_indices(len(dbl_vals), k=-1)
    dbl_vals[tril] = np.nan
    dbl_text[tril] = ""

    fig2 = go.Figure(go.Heatmap(
        z=dbl_vals,
        x=dbl_mat.columns.tolist(),
        y=dbl_mat.index.tolist(),
        colorscale="Reds",
        text=dbl_text,
        texttemplate="%{text}",
        hovertemplate="%{y} + %{x}<br>Doublets: %{z}<extra></extra>"
    ))
    fig2.update_layout(
        title=f"Doublet pair counts — {run_id}  ({len(doublets):,} doublets)",
        height=max(420, 22 * len(line_order)), xaxis_tickangle=45
    )
    display(fig2)

    # ── 3. Vireo probability distribution ────────────────────────────────
    prob_df = pd.concat([singlets, vireo_only]).copy()
    prob_df["vireo_prob_max"] = pd.to_numeric(prob_df["vireo_prob_max"], errors="coerce")

    fig3 = go.Figure()
    for line in line_order:
        sub = prob_df[prob_df["cell_line"] == line]["vireo_prob_max"].dropna()
        if len(sub) == 0:
            continue
        is_vireo = line.startswith("vireo:")
        fig3.add_trace(go.Violin(
            y=sub, name=line, box_visible=True,
            meanline_visible=True, points=False,
            line_color="#6d6875" if is_vireo else None
        ))
    fig3.update_layout(
        title=f"Vireo assignment probability per cell line — {run_id}",
        yaxis_title="prob_max (Vireo confidence)",
        height=420, showlegend=False
    )
    display(fig3)

    # ── 4. Vireo → DB concordance & gap bars ─────────────────────────────
    _MIN_CONCORDANCE = 0.80
    _MIN_GAP = 0.10
    if not dm.empty and "concordance" in dm.columns:
        dm["concordance"] = pd.to_numeric(dm["concordance"], errors="coerce")
        dm["gap"]         = pd.to_numeric(dm["gap"],         errors="coerce")
        dm["label"]       = dm["cell_line"].fillna(dm["assigned_line"])
        conf_color = {"high": "#2d6a4f", "low": "#f4a261",
                      "no_match": "#6d6875", "no_data": "#adb5bd"}
        bar_colors = [conf_color.get(str(c), "#adb5bd") for c in dm["confidence"]]

        fig4 = make_subplots(
            rows=1, cols=2,
            subplot_titles=("Best concordance per donor", "Gap (best − second best)"),
            horizontal_spacing=0.12
        )
        fig4.add_trace(go.Bar(
            x=dm["vireo_donor"], y=dm["concordance"],
            marker_color=bar_colors,
            text=dm["label"], textposition="outside",
            hovertemplate="%{x} → %{text}<br>Concordance: %{y:.3f}<extra></extra>"
        ), row=1, col=1)
        fig4.add_hline(y=_MIN_CONCORDANCE, line_dash="dash", line_color="grey",
                       annotation_text="min_concordance", row=1, col=1)

        fig4.add_trace(go.Bar(
            x=dm["vireo_donor"], y=dm["gap"],
            marker_color=bar_colors,
            hovertemplate="%{x}<br>Gap: %{y:.3f}<extra></extra>"
        ), row=1, col=2)
        fig4.add_hline(y=_MIN_GAP, line_dash="dash", line_color="grey",
                       annotation_text="min_gap", row=1, col=2)

        fig4.update_yaxes(row=1, col=1)
        fig4.update_layout(
            title=f"Vireo → DB donor matching — {run_id}",
            height=420, showlegend=False
        )
        display(fig4)
    else:
        print("donor_matches.tsv not found — skipping concordance plot.")


_scrna_busy = [False]

def render_scrna_cell(run_id):
    if _scrna_busy[0]:
        return
    _scrna_busy[0] = True
    try:
        clear_output(wait=False)
        display(demux_selector)
        render_scrna(run_id)
    finally:
        _scrna_busy[0] = False

demux_selector.observe(lambda ch: render_scrna_cell(ch['new']), names='value')

if demux_runs:
    render_scrna_cell(demux_runs[-1])


Dropdown(description='Demux run:', index=2, layout=Layout(width='350px'), options=('pool_ctr_v2', 'pool_ctr_v3…

### Validation — reference-guided Vireo (Pool_ctr)
Cell composition from `validation/donor_ids.tsv` — reference-guided Vireo run on Pool_ctr
using known cell-line genotypes. Donor labels are the actual cell-line names (lowercase in file).

In [8]:
_CELL_LINE_DISPLAY = {
    "rko": "RKO", "hct116": "HCT116", "ht29": "HT29",
    "lim1215": "LIM1215", "caco2": "CACO2", "hct8": "HCT8", "dld1": "DLD1",
    "doublet": "doublet", "unassigned": "unassigned",
}

_val_path = "validation/donor_ids.tsv"
val = pd.read_csv(_val_path, sep="\t")

# Normalise donor_id display label
val["donor_label"] = val["donor_id"].map(
    lambda x: _CELL_LINE_DISPLAY.get(str(x).lower(), str(x))
)
val["prob_max"] = pd.to_numeric(val["prob_max"], errors="coerce")
val["n_vars"]   = pd.to_numeric(val["n_vars"],   errors="coerce")

# Order: singlets by count desc, then doublet, unassigned
_meta = {"doublet", "unassigned"}
_singlet_order = (
    val[~val["donor_label"].isin(_meta)]
    .groupby("donor_label").size()
    .sort_values(ascending=False).index.tolist()
)
_donor_order = _singlet_order + ["doublet", "unassigned"]

# ── 1. Cell composition bar ───────────────────────────────────────────────
_comp = val["donor_label"].value_counts().reindex(_donor_order, fill_value=0)
_bar_colors = [
    "#adb5bd" if d == "unassigned" else
    "#e63946" if d == "doublet"    else
    "#2d6a4f"
    for d in _donor_order
]

fig_v1 = go.Figure(go.Bar(
    x=_comp.index, y=_comp.values,
    marker_color=_bar_colors,
    hovertemplate="%{x}<br>Cells: %{y:,}<extra></extra>",
    text=_comp.values, textposition="outside"
))
fig_v1.update_layout(
    title=f"Cell composition — Pool_ctr reference-guided Vireo  ({len(val):,} cells total)",
    xaxis_title="Donor / status", yaxis_title="Cell count",
    height=420, showlegend=False
)
fig_v1.show()

# ── 2. prob_max violin per donor ──────────────────────────────────────────
fig_v2 = go.Figure()
for donor in _singlet_order:
    sub = val[val["donor_label"] == donor]["prob_max"].dropna()
    if len(sub) == 0:
        continue
    fig_v2.add_trace(go.Violin(
        y=sub, name=donor,
        box_visible=True, meanline_visible=True, points=False
    ))
fig_v2.update_layout(
    title="Vireo assignment probability (prob_max) per cell line — Pool_ctr",
    yaxis_title="prob_max", height=420, showlegend=False
)
fig_v2.show()

# ── 3. n_vars distribution per donor ─────────────────────────────────────
fig_v3 = go.Figure()
for donor in _singlet_order:
    sub = val[val["donor_label"] == donor]["n_vars"].dropna()
    if len(sub) == 0:
        continue
    fig_v3.add_trace(go.Violin(
        y=sub, name=donor,
        box_visible=True, meanline_visible=True, points=False
    ))
fig_v3.update_layout(
    title="Number of informative variants per cell — Pool_ctr",
    yaxis_title="n_vars (SNPs used for assignment)", height=420, showlegend=False
)
fig_v3.show()

# Summary table
_summary_v = (
    val.groupby("donor_label")
    .agg(
        n_cells=("cell", "count"),
        mean_prob_max=("prob_max", "mean"),
        median_prob_max=("prob_max", "median"),
        mean_n_vars=("n_vars", "mean"),
    )
    .reindex(_donor_order)
    .reset_index()
)
display(_summary_v.style.format({
    "mean_prob_max":   "{:.3f}",
    "median_prob_max": "{:.3f}",
    "mean_n_vars":     "{:.0f}",
}))


,donor_label,n_cells,mean_prob_max,median_prob_max,mean_n_vars
0,RKO,4610,0.999,1.000,669
1,HCT116,2976,0.999,1.000,482
2,HT29,817,0.999,1.000,573
3,HCT8,119,0.984,0.996,451
4,CACO2,87,0.996,1.000,631
5,DLD1,82,0.979,0.995,428
6,LIM1215,30,0.996,1.000,462
7,doublet,575,0.003,0.000,823
8,unassigned,1012,0.489,0.476,75


## Concordance profiles
Full concordance (ALT-recall) of each Vireo donor against every DB reference.
Computes the matrix live from the Vireo VCF — takes ~20 s on first load per run.

**Plot A** — side-by-side bar chart: best-matching vs worst-matching donor against all DB references.  
**Plot B** — horizontal bar charts for three auto-selected donors (best / median / worst).  
**Plot C** — distribution histogram: how concordance scores are spread across all DB references for those same three donors. A good match shows as a clear spike above the threshold (red dashed line); a non-matching donor trails off to zero at the high end.  

**Colour key**: green = high-confidence match · orange = assigned but low-confidence · light grey = insufficient shared positions · medium grey = unmatched.

In [9]:
import sys as _sys
_sys.path.insert(0, 'scripts')
from match_vireo import load_donor_genotypes, load_db_genotypes, compute_concordance as _compute_concordance

DB_PATH = 'results/variants.db'
MIN_CONCORDANCE = 0.80
MIN_POS = 200

# ── Reference label lookup ────────────────────────────────────────────────
_ref_labels_cache = {}
def get_ref_labels(run_ids):
    if not _ref_labels_cache:
        rows = query("""
            SELECT r.run_id, COALESCE(s.cell_line, s.name) AS label
            FROM runs r JOIN samples s ON r.sample_id = s.sample_id
        """)
        _ref_labels_cache.update(dict(zip(rows['run_id'], rows['label'])))
    return [_ref_labels_cache.get(r, r) for r in run_ids]

# ── Concordance computation (cached per run) ──────────────────────────────
_conc_cache = {}

def compute_conc(run_id):
    if run_id in _conc_cache:
        return _conc_cache[run_id]
    vireo_dir = f'results/demux/{run_id}/vireo'
    matches_path = f'results/demux/{run_id}/donor_matches.tsv'
    print('Computing concordance matrix … (first load per run, ~20 s)')
    donors, positions, vireo_dosage = load_donor_genotypes(vireo_dir)
    ref_ids, db_dosage = load_db_genotypes(DB_PATH, ['all'], positions)
    concordance, n_shared = _compute_concordance(vireo_dosage, db_dosage)
    dm = pd.read_csv(matches_path, sep='\t')
    ref_labels = get_ref_labels(ref_ids)
    _conc_cache[run_id] = (donors, ref_ids, ref_labels, concordance, n_shared, dm)
    return _conc_cache[run_id]

# ── Render ────────────────────────────────────────────────────────────────
conc_selector = widgets.Dropdown(
    options=demux_runs,
    description='Demux run:',
    layout=widgets.Layout(width='350px')
)
_conc_busy = [False]

def render_concordance(run_id):
    if _conc_busy[0]:
        return
    _conc_busy[0] = True
    try:
        clear_output(wait=False)
        display(conc_selector)

        donors, ref_ids, ref_labels, concordance, n_shared, dm = compute_conc(run_id)
        n_donors = len(donors)
        n_refs   = len(ref_ids)

        # Best concordance per donor (NaN → 0 for ranking)
        best_c = np.array([
            np.nanmax(concordance[di]) if not np.all(np.isnan(concordance[di])) else 0.0
            for di in range(n_donors)
        ])

        # Resolve donor → assigned reference + confidence from donor_matches.tsv
        dm_indexed = dm.set_index('vireo_donor')
        def donor_match_info(donor):
            if donor not in dm_indexed.index:
                return None, None
            row = dm_indexed.loc[donor]
            return row.get('assigned_line', None), row.get('confidence', None)

        # ── Plot A: side-by-side best match vs worst match ──────────────────
        best_di  = int(np.argmax(best_c))
        worst_di = int(np.argmin(best_c))

        def make_donor_bars(di):
            c_row = concordance[di].copy()
            invalid = np.isnan(c_row) | (n_shared[di] < MIN_POS)
            display_c = np.where(invalid, 0.0, c_row)
            order = np.argsort(display_c)[::-1]
            sorted_c      = display_c[order]
            sorted_labels = [ref_labels[i] for i in order]
            sorted_ids    = [ref_ids[i]    for i in order]
            assigned, conf = donor_match_info(donors[di])
            colors = []
            for pos, (orig_idx, rid) in enumerate(zip(order, sorted_ids)):
                if rid == assigned and conf == 'high':
                    colors.append('#2d6a4f')
                elif rid == assigned and conf == 'low':
                    colors.append('#f4a261')
                elif invalid[orig_idx]:
                    colors.append('#e9ecef')
                else:
                    colors.append('#adb5bd')
            return sorted_c, sorted_labels, colors, assigned, conf

        fig_a = make_subplots(
            rows=1, cols=2,
            subplot_titles=(
                f'{donors[best_di]} — best match',
                f'{donors[worst_di]} — worst / no match'
            ),
            horizontal_spacing=0.12
        )
        fig_a_labels = []
        for col_idx, di in enumerate([best_di, worst_di], start=1):
            c_vals, labels, colors, assigned, conf = make_donor_bars(di)
            fig_a_labels.append(labels)
            fig_a.add_trace(go.Bar(
                x=labels, y=c_vals,
                marker_color=colors,
                hovertemplate='%{x}<br>Concordance: %{y:.3f}<extra></extra>',
                showlegend=False
            ), row=1, col=col_idx)
            fig_a.add_hline(y=MIN_CONCORDANCE, line_dash='dash', line_color='#e63946',
                            annotation_text='threshold', row=1, col=col_idx)

        fig_a.update_yaxes(range=[0, 1], title_text='Concordance')
        fig_a.update_layout(height=480, title=f'Match vs no-match — {run_id}')

        # Always label the first bar (best/closest match), thin the rest to ~8 visible
        for col_idx, labels in enumerate(fig_a_labels, start=1):
            step = max(1, len(labels) // 8)
            tick_indices = list(range(0, len(labels), step))
            tickvals = [labels[i] for i in tick_indices]
            axis_key = 'xaxis' if col_idx == 1 else 'xaxis2'
            fig_a.update_layout(**{axis_key: dict(
                tickmode='array', tickvals=tickvals, ticktext=tickvals, tickangle=45
            )})
        display(fig_a)

        # ── Plot B: 3-donor concordance profiles ─────────────────────────────
        sorted_by_c = np.argsort(best_c)
        worst_pick  = int(sorted_by_c[0])
        best_pick   = int(sorted_by_c[-1])
        median_pick = int(sorted_by_c[len(sorted_by_c) // 2])
        trio = [best_pick, median_pick, worst_pick]
        trio_labels = ['Best match', 'Median match', 'Worst / no match']

        fig_b = make_subplots(
            rows=3, cols=1,
            subplot_titles=[
                f'{tl} — {donors[di]} (best concordance {best_c[di]:.3f})'
                for tl, di in zip(trio_labels, trio)
            ],
            vertical_spacing=0.06,
            shared_xaxes=True
        )
        for row_idx, di in enumerate(trio, start=1):
            c_vals, labels, colors, _, _ = make_donor_bars(di)
            fig_b.add_trace(go.Bar(
                x=c_vals[::-1],
                y=labels[::-1],
                orientation='h',
                marker_color=colors[::-1],
                hovertemplate='%{y}<br>Concordance: %{x:.3f}<extra></extra>',
                showlegend=False
            ), row=row_idx, col=1)
            fig_b.add_vline(x=MIN_CONCORDANCE, line_dash='dash', line_color='#e63946',
                            row=row_idx, col=1)

        fig_b.update_xaxes(range=[0, 1], title_text='Concordance', row=3, col=1)
        fig_b.update_yaxes(tickmode='linear', dtick=1)
        fig_b.update_layout(
            height=max(600, 20 * n_refs * 3),
            title=f'Concordance profiles — {run_id}  (green = high match, orange = low match, grey = unmatched)'
        )
        display(fig_b)

        # ── Plot C: concordance score distribution ───────────────────────────
        BINS = np.linspace(0, 1, 21)   # 20 bins of width 0.05

        fig_c = make_subplots(
            rows=1, cols=3,
            subplot_titles=[
                f'{donors[di]}  ({trio_labels[idx]})'
                for idx, di in enumerate(trio)
            ],
            horizontal_spacing=0.08
        )
        for col_idx, (idx, di) in enumerate(zip(range(3), trio), start=1):
            c_row   = concordance[di]
            valid_m = ~np.isnan(c_row) & (n_shared[di] >= MIN_POS)
            c_valid = c_row[valid_m]

            counts, edges = np.histogram(c_valid, bins=BINS)
            ymax = max(1, int(counts.max()))

            fig_c.add_trace(go.Bar(
                x=(edges[:-1] + edges[1:]) / 2,
                y=counts,
                width=(edges[1] - edges[0]) * 0.85,
                marker_color='#adb5bd',
                showlegend=False,
                hovertemplate='Concordance %{customdata[0]:.2f}–%{customdata[1]:.2f}: '
                              '%{y} references<extra></extra>',
                customdata=np.column_stack([edges[:-1], edges[1:]])
            ), row=1, col=col_idx)

            # MIN_CONCORDANCE threshold line
            fig_c.add_trace(go.Scatter(
                x=[MIN_CONCORDANCE, MIN_CONCORDANCE], y=[0, ymax * 1.2],
                mode='lines', line=dict(color='#e63946', dash='dash', width=1.5),
                showlegend=False, hoverinfo='skip'
            ), row=1, col=col_idx)

            # Assigned match marker
            assigned_c, conf_c = donor_match_info(donors[di])
            if assigned_c and assigned_c != 'no_match' and assigned_c in ref_ids:
                ari     = ref_ids.index(assigned_c)
                match_c = float(c_row[ari]) if not np.isnan(c_row[ari]) else None
                if match_c is not None:
                    line_color = '#2d6a4f' if conf_c == 'high' else '#f4a261'
                    fig_c.add_trace(go.Scatter(
                        x=[match_c, match_c], y=[0, ymax * 1.2],
                        mode='lines', line=dict(color=line_color, width=2),
                        showlegend=False, hoverinfo='skip'
                    ), row=1, col=col_idx)

        fig_c.update_xaxes(range=[0, 1], title_text='Concordance score')
        fig_c.update_yaxes(title_text='# DB references', row=1, col=1)
        fig_c.update_layout(
            height=320,
            title=(
                f'Concordance distribution across all DB references — {run_id}  |  '
                'red dashed = threshold  |  green/orange line = assigned match'
            )
        )
        display(fig_c)
    finally:
        _conc_busy[0] = False

conc_selector.observe(lambda ch: render_concordance(ch['new']), names='value')
if demux_runs:
    render_concordance(demux_runs[-1])


Dropdown(description='Demux run:', index=2, layout=Layout(width='350px'), options=('pool_ctr_v2', 'pool_ctr_v3…

## STAR alignment QC
Spliced read percentage and short-read fraction across all runs (from STAR log).

In [10]:
star_qc = query("""
    SELECT r.run_id, s.name AS sample,
           r.mapping_rate
    FROM runs r
    JOIN samples s ON r.sample_id = s.sample_id
    ORDER BY r.run_date DESC, r.run_id
""")

# Try to pull splice/short metrics from STAR logs directly
import re

def parse_star_log(run_id, sample):
    log_path = f"results/{run_id}/star/{sample}/Log.final.out"
    if not os.path.exists(log_path):
        return None, None
    text = open(log_path).read()
    m_splice = re.search(r"% of reads mapped to multiple loci.*?([\d.]+)%", text)
    m_short  = re.search(r"% of reads unmapped: too short.*?([\d.]+)%", text)
    m_annot  = re.search(r"% of splices: Annotated \(sjdb\).*?([\d.]+)%", text)
    splice = float(m_annot.group(1)) if m_annot else None
    short  = float(m_short.group(1))  if m_short  else None
    return splice, short

star_qc = query("""
    SELECT r.run_id, s.name AS sample, r.mapping_rate
    FROM runs r JOIN samples s ON r.sample_id = s.sample_id
    ORDER BY r.run_id
""")

splice_vals, short_vals = [], []
for _, row in star_qc.iterrows():
    sp, sh = parse_star_log(row["run_id"], row["sample"])
    splice_vals.append(sp)
    short_vals.append(sh)

star_qc["annotated_splice_pct"] = splice_vals
star_qc["pct_too_short"] = short_vals

has_star_data = star_qc["annotated_splice_pct"].notna().any()

if has_star_data:
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Annotated splice % (higher = better)",
                        "% reads too short (lower = better)"),
        horizontal_spacing=0.12
    )
    fig.add_trace(go.Bar(
        x=star_qc["sample"],
        y=star_qc["annotated_splice_pct"],
        marker_color="#2d6a4f"
    ), row=1, col=1)
    fig.add_trace(go.Bar(
        x=star_qc["sample"],
        y=star_qc["pct_too_short"],
        marker_color="#e63946"
    ), row=1, col=2)
    fig.update_layout(height=380, showlegend=False, title="STAR alignment QC")
    fig.show()
else:
    print("STAR log files not found or metrics not yet parsed.")
    print("Run the pipeline first, then re-execute this cell.")

display(star_qc)

STAR log files not found or metrics not yet parsed.
Run the pipeline first, then re-execute this cell.


,run_id,sample,mapping_rate,annotated_splice_pct,pct_too_short
0,SRR5071654_hg38,SRR5071654,100.0,None,1.52
1,SRR5071655_hg38,SRR5071655,100.0,None,1.57
2,SRR5071656_hg38,SRR5071656,100.0,None,1.83
3,SRR5071657_hg38,SRR5071657,100.0,None,1.51
4,SRR5071658_hg38,SRR5071658,100.0,None,1.77
5,SRR5071659_hg38,SRR5071659,100.0,None,1.67
6,SRR5071660_hg38,SRR5071660,100.0,None,1.31
7,SRR5071661_hg38,SRR5071661,100.0,None,1.45
8,SRR5071662_hg38,SRR5071662,100.0,None,1.28
9,SRR5071663_hg38,SRR5071663,100.0,None,1.77


## Raw SQL explorer
Run any query against the database.

In [11]:
sql_box = widgets.Textarea(
    value="SELECT s.name, r.run_id, r.mapping_rate, r.total_filt\nFROM runs r JOIN samples s ON r.sample_id = s.sample_id\nORDER BY r.run_date DESC;",
    layout=widgets.Layout(width='100%', height='100px')
)
run_btn = widgets.Button(description='Run query', button_style='primary')

def on_run_query(btn):
    clear_output(wait=False)
    display(sql_box, run_btn)
    try:
        result = query(sql_box.value)
        display(result)
    except Exception as e:
        print(f'Error: {e}')

run_btn.on_click(on_run_query)
display(sql_box, run_btn)


Textarea(value='SELECT s.name, r.run_id, r.mapping_rate, r.total_filt\nFROM runs r JOIN samples s ON r.sample_…

Button(button_style='primary', description='Run query', style=ButtonStyle())